In [4]:
import sys
import pandas as pd
from pathlib import Path

# Projenin ana dizinini Python yoluna ekle
PROJECT_ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Kendi modülümüzü içe aktar
from tarimsal_hastalik.data_collection.weather_client import get_hourly_weather

print("Proje yolu:", PROJECT_ROOT)
print("Weather client başarıyla içe aktarıldı!")

Proje yolu: c:\Users\HELO\TarımsalHastalıkErkenUyarı
Weather client başarıyla içe aktarıldı!


In [2]:
# Niğde için Nisan 2026 meteorolojik verileri
df = get_hourly_weather(
    latitude=37.97,
    longitude=34.68,
    start_date="2026-04-01",
    end_date="2026-04-30",
)

print("Veri başarıyla alındı!")
print("Toplam kayıt:", len(df))

display(df.head(10))

Veri başarıyla alındı!
Toplam kayıt: 720


,time,temperature_2m,relative_humidity_2m,precipitation
0,2026-04-01 00:00:00,6.5,81,0.0
1,2026-04-01 01:00:00,5.8,83,0.0
2,2026-04-01 02:00:00,4.1,87,0.0
3,2026-04-01 03:00:00,3.8,89,0.0
4,2026-04-01 04:00:00,3.0,91,0.0
5,2026-04-01 05:00:00,2.6,88,0.0
6,2026-04-01 06:00:00,2.4,85,0.0
7,2026-04-01 07:00:00,3.0,82,0.0
8,2026-04-01 08:00:00,6.9,69,0.0
9,2026-04-01 09:00:00,9.6,56,0.0


In [5]:
# 1. Eksik değer kontrolü
print("Eksik değerler:")
display(df.isnull().sum())

# 2. Tekrarlanan zaman kayıtları
print("Tekrarlanan zaman kayıtları:", df["time"].duplicated().sum())

# 3. Saatlik zaman aralığı kontrolü
time_differences = df["time"].diff().dropna()

print(
    "Saatlik aralık dışında kalan kayıtlar:",
    (time_differences != pd.Timedelta(hours=1)).sum()
)

# 4. Değer aralığı kontrolleri
print("Bağıl nem 0-100 dışında:",
      (~df["relative_humidity_2m"].between(0, 100)).sum())

print("Negatif yağış kaydı:",
      (df["precipitation"] < 0).sum())

print("Sıcaklık -50 ile 60°C dışında:",
      (~df["temperature_2m"].between(-50, 60)).sum())

Eksik değerler:


time                    0
temperature_2m          0
relative_humidity_2m    0
precipitation           0
dtype: int64

Tekrarlanan zaman kayıtları: 0
Saatlik aralık dışında kalan kayıtlar: 0
Bağıl nem 0-100 dışında: 0
Negatif yağış kaydı: 0
Sıcaklık -50 ile 60°C dışında: 0


In [6]:
print("İlk kayıt:", df["time"].min())
print("Son kayıt:", df["time"].max())
print("Toplam kayıt:", len(df))

expected_records = 30 * 24

print("Beklenen kayıt:", expected_records)
print("Kayıt sayısı doğru mu:", len(df) == expected_records)

İlk kayıt: 2026-04-01 00:00:00
Son kayıt: 2026-04-30 23:00:00
Toplam kayıt: 720
Beklenen kayıt: 720
Kayıt sayısı doğru mu: True


In [7]:
from pathlib import Path

# Ham veri klasörünü oluştur
raw_dir = PROJECT_ROOT / "data" / "raw" / "weather" / "open_meteo"
raw_dir.mkdir(parents=True, exist_ok=True)

# Dosya yolu
raw_file = raw_dir / "nigde_2026-04.csv"

# CSV olarak kaydet
df.to_csv(raw_file, index=False, encoding="utf-8")

# Kayıt kontrolü
print("Dosya kaydedildi:", raw_file)
print("Dosya mevcut mu:", raw_file.exists())
print("Dosya boyutu (KB):", round(raw_file.stat().st_size / 1024, 2))

Dosya kaydedildi: c:\Users\HELO\TarımsalHastalıkErkenUyarı\data\raw\weather\open_meteo\nigde_2026-04.csv
Dosya mevcut mu: True
Dosya boyutu (KB): 22.88


In [8]:
import json

# Metadata dosyasını oku
metadata_file = raw_dir / "metadata.json"

with open(metadata_file, "r", encoding="utf-8") as file:
    metadata = json.load(file)

# CSV verisiyle karşılaştır
print("Metadata geçerli: True")
print("Metadata kayıt sayısı:", metadata["record_count"])
print("CSV kayıt sayısı:", len(df))
print("Kayıt sayıları eşleşiyor:",
      metadata["record_count"] == len(df))

print("Veri kaynağı:", metadata["source"]["provider"])
print("Veri dönemi:",
      metadata["period"]["start_date"],
      "-",
      metadata["period"]["end_date"])

Metadata geçerli: True
Metadata kayıt sayısı: 720
CSV kayıt sayısı: 720
Kayıt sayıları eşleşiyor: True
Veri kaynağı: Open-Meteo
Veri dönemi: 2026-04-01 - 2026-04-30
